In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
import xarray as xr
from concurrent.futures import TimeoutError

tqdm.pandas()

def random_date(start_date, end_date):
    """Generate a random date between start_date and end_date."""
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=random_days)

def load_terraclimate_dataset():
    print("Opening catalog...")
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    print("Catalog opened.")
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]
    print("Asset retrieved.")
    if "xarray:storage_options" in asset.extra_fields:
        print("Opening Zarr...")
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        print("Opening dataset...")
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )
    print("Dataset loaded.")
    return ds

print("Loading dataset...")
ds = load_terraclimate_dataset()

time_slice = slice("2000-01-01", "2007-12-31")
print("Selecting slices...")
ds_pet = ds['pet'].sel(time=time_slice)
ds_def = ds['def'].sel(time=time_slice)
ds_aet = ds['aet'].sel(time=time_slice)
print("Slices selected.")

np.random.seed(42)  
num_samples = 8000
lats = np.random.uniform(-90, 90, num_samples)
lons = np.random.uniform(-180, 180, num_samples)
start_date = datetime(2000, 1, 1)
end_date = datetime(2007, 12, 31)
dates = [random_date(start_date, end_date) for _ in range(num_samples)]

df = pd.DataFrame({
    'Latitude': lats,
    'Longitude': lons,
    'Sample Date': dates
})

def assign_terraclimate_values(row):
    try:
        lat = row['Latitude']
        lon = row['Longitude']
        date = pd.to_datetime(row['Sample Date'])
        nearest_pet = ds_pet.sel(time=date, lat=lat, lon=lon, method='nearest')
        nearest_def = ds_def.sel(time=date, lat=lat, lon=lon, method='nearest')
        nearest_aet = ds_aet.sel(time=date, lat=lat, lon=lon, method='nearest')
        pet = float(nearest_pet.values) if not np.isnan(nearest_pet.values) else np.nan
        def_val = float(nearest_def.values) if not np.isnan(nearest_def.values) else np.nan
        aet = float(nearest_aet.values) if not np.isnan(nearest_aet.values) else np.nan
        return pd.Series({
            'Pet': pet,
            'Def': def_val,
            'Aet': aet
        })
    except Exception as e:
        print(f"Error for row: {e}")
        return pd.Series({
            'Pet': np.nan,
            'Def': np.nan,
            'Aet': np.nan
        })

print("Assigning values sequentially...")
terraclimate_features = df.progress_apply(assign_terraclimate_values, axis=1)

df_with_features = pd.concat([df[['Longitude', 'Latitude', 'Sample Date']], terraclimate_features], axis=1)

df_with_features.to_csv('terraclimate_2000_2007_8000_samples.csv', index=False)
print("Processing complete.")

Loading dataset...
Opening catalog...
Catalog opened.
Asset retrieved.
Opening dataset...
Dataset loaded.
Selecting slices...
Slices selected.
Assigning values sequentially...


  0%|          | 13/8000 [34:03<348:40:21, 157.16s/it]


KeyboardInterrupt: 